# 1. Data Acquisition.

In [1]:
from data_pipeline import load_snapshot
prices, sectors, vix = load_snapshot()
print(prices.shape, prices.index.min(), prices.index.max())
vix.tail()

(5736, 499) 2004-01-02 00:00:00 2026-10-20 00:00:00


Date
2026-10-14    16.870001
2026-10-15    16.459999
2026-10-16    19.990000
2026-10-19    18.440001
2026-10-20    17.309999
Name: VIX, dtype: float64

# 2. Data Cleaning and Return Calculation

In [2]:
import numpy as np
import pandas as pd

# Align the VIX to stock trading days. ffill carries the last known value forward
# (no look-ahead: a day's VIX is never filled with a later value).
vix = vix.reindex(prices.index).ffill()
print("VIX missing after alignment:", vix.isna().sum())

# Daily returns. fill_method=None means gaps stay NaN instead of being silently filled.
daily_ret = prices.pct_change(fill_method=None)

# Sanity check: the biggest single-day moves (look for obvious data errors)
print(daily_ret.abs().stack().dropna().sort_values().tail(5))

VIX missing after alignment: 0
Date            
2019-01-24  PCG     0.745933
2008-10-29  LVS     0.800000
2008-10-13  MS      0.869835
2008-12-05  HIG     1.023578
2026-08-19  MRNA    1.769695
dtype: float64


# 3. Strategy: signal construction

In [3]:
# Fixed dates, decided in design_choices.md BEFORE looking at results
DEV_START, DEV_END = "2005-01-01", "2017-12-31"
TEST_START = "2018-01-01"   # do NOT use anything from here on until Task 7

# 5-day return ending on each day (uses only prices up to and including that day)
ret5 = prices.pct_change(5, fill_method=None)

# Subtract each stock's sector average that same day -> "sector-adjusted" return
sector_map = sectors.set_index("ticker")["sector"].reindex(prices.columns)
sector_mean = ret5.T.groupby(sector_map).transform("mean").T
signal = ret5 - sector_mean

# Weekly rebalance dates = the last trading day of each week
reb_dates = pd.DatetimeIndex(
    prices.index.to_series().groupby(prices.index.to_period("W")).last().values
)
print(signal.shape, "| weekly dates:", len(reb_dates))
signal.loc[reb_dates].dropna(how="all").iloc[-1].sort_values().head()

(5736, 499) | weekly dates: 1191


SKYD   -0.236050
FLEX   -0.168072
BE     -0.160709
COHR   -0.157758
P      -0.151109
Name: 2026-10-20 00:00:00, dtype: float64

In [4]:
prices.loc['2026-08-17':'2026-08-21', 'MRNA']

Date
2026-08-17     64.459999
2026-08-18     62.959999
2026-08-19    174.380005
2026-08-20    133.320007
2026-08-21    145.130005
Name: MRNA, dtype: float64

# 4. Backtest (development period only: 2005-2017)

In [5]:
def run_reversal(signal, prices, reb_dates, end, cost_bps=5.0, q=0.10):
    """Weekly long/short reversal. Signal at close of day t, trade at close of day t+1."""
    idx = prices.index
    pos = idx.get_indexer(reb_dates)
    assert (pos >= 0).all(), "rebalance date is not a trading day"
    trade_pos = pos + 1                                   # trade ONE day after the signal
    capped = idx[np.minimum(trade_pos, len(idx) - 1)]
    keep = (trade_pos < len(idx)) & (capped <= pd.Timestamp(end))   # never use prices after `end`
    reb_dates, trade_pos = reb_dates[keep], trade_pos[keep]
    trade_dates = idx[trade_pos]
    assert (trade_dates > reb_dates).all(), "look-ahead: trade must be AFTER the signal date"

    w_prev = pd.Series(0.0, index=prices.columns)
    rows = []
    for k in range(len(reb_dates) - 1):
        s = signal.loc[reb_dates[k]]
        s = s[s.notna() & prices.iloc[trade_pos[k]].notna()]   # only stocks tradable on the trade date
        n = int(len(s) * q)                                    # decile size
        assert n >= 5, f"too few stocks on {reb_dates[k]}"
        w = pd.Series(0.0, index=prices.columns)
        w[s.nsmallest(n).index] = 1.0 / n                      # long the biggest losers vs sector
        w[s.nlargest(n).index] = -1.0 / n                      # short the biggest winners vs sector
        assert abs(w.sum()) < 1e-9                             # dollar-neutral
        r = (prices.iloc[trade_pos[k + 1]] / prices.iloc[trade_pos[k]] - 1).fillna(0.0)
        gross = float((w * r).sum())
        turnover = float((w - w_prev).abs().sum())             # dollars traded (both sides)
        cost = turnover * cost_bps / 1e4
        rows.append((trade_dates[k + 1], gross, cost, gross - cost, n, turnover))
        w_prev = w
    cols = ["end_date", "gross", "cost", "net", "n_per_side", "turnover"]
    return pd.DataFrame(rows, columns=cols).set_index("end_date")


def perf(r, periods=52):
    r = r.dropna()
    wealth = (1 + r).cumprod()
    ann_ret, ann_vol = r.mean() * periods, r.std() * np.sqrt(periods)
    return pd.Series({"cum_return": wealth.iloc[-1] - 1, "ann_return": ann_ret,
                      "ann_vol": ann_vol, "sharpe": ann_ret / ann_vol,
                      "max_drawdown": (wealth / wealth.cummax() - 1).min(), "weeks": len(r)})

In [6]:
dev_reb = reb_dates[(reb_dates >= DEV_START) & (reb_dates <= DEV_END)]

res5  = run_reversal(signal, prices, dev_reb, end=DEV_END, cost_bps=5)
res10 = run_reversal(signal, prices, dev_reb, end=DEV_END, cost_bps=10)

results = pd.DataFrame({
    "gross (no costs)": perf(res5["gross"]),
    "net, 5 bps":       perf(res5["net"]),
    "net, 10 bps":      perf(res10["net"]),
})
print(results.round(3))
print("avg stocks per side:", round(res5["n_per_side"].mean(), 1),
      "| avg weekly turnover:", round(res5["turnover"].mean(), 2))

              gross (no costs)  net, 5 bps  net, 10 bps
cum_return               3.337       0.380       -0.562
ann_return               0.124       0.036       -0.052
ann_vol                  0.152       0.152        0.152
sharpe                   0.816       0.237       -0.342
max_drawdown            -0.205      -0.244       -0.620
weeks                  676.000     676.000      676.000
avg stocks per side: 42.6 | avg weekly turnover: 3.39
